# 01 — S&P 500 Universe Panel

**Goal:** Build a firm-year panel of S&P 500 historical membership for 2012–2024
(extending one year beyond the sample window to support portfolio formation in 2013
and to capture post-sample years for robustness).

**Output:** `data/sp500_universe.parquet`
- One row per (year, ticker)
- 6,535 firm-year observations
- 751 distinct tickers over the window (≈502 per year on average, with attrition
  contributing ~250 additional historical members)
- Survivorship-bias-free: includes firms that exited the index during the window
  (e.g., Twitter after the Musk acquisition in 2022)

**Data source:** GitHub repository `fja05680/sp500` (Aultman et al., 2026), which maintains
point-in-time S&P 500 membership by extending a published historical base list
with index additions and deletions tracked against public sources. The dataset provides 879 snapshots over the 2012–2024 window. The current
notebook extracts the latest snapshot on or before each December 31 as the year-end
membership list.

**Why this source:** WU Vienna's WRDS subscription does not include the CRSP
historical index tables (`crsp_a_indexes` schema is permission-denied). Capital IQ's
web-based Companies screener does not expose point-in-time index membership as a
queryable criterion at WU's subscription tier.

**Sanity checks performed:**
- Snapshot row counts (each year-end snapshot contains 497–506 tickers, consistent
  with the S&P 500's actual constituent count, which periodically exceeds 500 due
  to dual share classes such as GOOG/GOOGL and FOX/FOXA)
- Known ticker transitions verified: Facebook → Meta (FB present 2013–2021, META
  from 2022 onward); Yahoo → Altaba (AABA, the dataset's forward-corrected ticker

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
!pip install pyarrow

In [3]:
DATA_RAW = Path.home() / "thesis" / "data" / "raw"
DATA_PROCESSED = Path.home() / "thesis" / "data"

raw = pd.read_csv(DATA_RAW / "sp500_historical_components.csv")

print(f"Rows: {len(raw)}")
print(f"Columns: {list(raw.columns)}")
print(f"Date range: {raw['date'].min()} to {raw['date'].max()}")
raw.head(3)

Rows: 2705
Columns: ['date', 'tickers']
Date range: 1996-01-02 to 2026-01-14


,date,tickers
0,1996-01-02,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
1,1996-01-03,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
2,1996-01-04,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."


In [4]:
# Convert date to datetime
raw["date"] = pd.to_datetime(raw["date"])

# Split the comma-separated tickers string into a list
raw["tickers"] = raw["tickers"].str.split(",")

# Verify the parsing worked
print(f"First snapshot has {len(raw['tickers'].iloc[0])} tickers")
print(f"Last snapshot has {len(raw['tickers'].iloc[-1])} tickers")
print(f"First five tickers in first snapshot: {raw['tickers'].iloc[0][:5]}")
print(f"First five tickers in last snapshot:  {raw['tickers'].iloc[-1][:5]}")

First snapshot has 487 tickers
Last snapshot has 503 tickers
First five tickers in first snapshot: ['AAL', 'AAMRQ', 'AAPL', 'ABI', 'ABS']
First five tickers in last snapshot:  ['A', 'AAPL', 'ABBV', 'ABNB', 'ABT']


In [5]:
sample_start = pd.Timestamp("2012-01-01")
sample_end = pd.Timestamp("2024-06-30")

sample = raw[(raw["date"] >= sample_start) & (raw["date"] <= sample_end)].copy()
sample = sample.reset_index(drop=True)

print(f"Snapshots in sample window: {len(sample)}")
print(f"First snapshot: {sample['date'].min().date()}")
print(f"Last snapshot:  {sample['date'].max().date()}")
print(f"Tickers in earliest snapshot: {len(sample['tickers'].iloc[0])}")
print(f"Tickers in latest snapshot:   {len(sample['tickers'].iloc[-1])}")

Snapshots in sample window: 879
First snapshot: 2012-01-03
Last snapshot:  2024-06-24
Tickers in earliest snapshot: 497
Tickers in latest snapshot:   503


In [6]:
exploded = sample.explode("tickers").rename(columns={"tickers": "ticker"})
exploded["ticker"] = exploded["ticker"].str.strip()
exploded = exploded.reset_index(drop=True)

print(f"Rows after exploding: {len(exploded):,}")
print(f"Unique tickers: {exploded['ticker'].nunique()}")
print(f"Unique dates: {exploded['date'].nunique()}")
exploded.head()

Rows after exploding: 441,167
Unique tickers: 770
Unique dates: 879


,date,ticker
0,2012-01-03,A
1,2012-01-03,AABA
2,2012-01-03,AAPL
3,2012-01-03,ABC
4,2012-01-03,ABT


In [7]:
year_ends = pd.DataFrame({
    "year": range(2012, 2025),  # 2012 to 2024 inclusive
    "year_end_date": pd.to_datetime([f"{y}-12-31" for y in range(2012, 2025)])
})

# For each year-end, find the latest snapshot on or before that date
year_end_snapshots = []
for _, row in year_ends.iterrows():
    snapshot_date = sample[sample["date"] <= row["year_end_date"]]["date"].max()
    matching = exploded[exploded["date"] == snapshot_date].copy()
    matching["year"] = row["year"]
    year_end_snapshots.append(matching)

panel = pd.concat(year_end_snapshots, ignore_index=True)
panel = panel[["year", "ticker", "date"]].rename(columns={"date": "snapshot_date"})

print(f"Total firm-year rows: {len(panel):,}")
print(f"Unique tickers (firms) across all years: {panel['ticker'].nunique()}")
print()
print("Tickers per year:")
print(panel.groupby("year")["ticker"].nunique())

Total firm-year rows: 6,535
Unique tickers (firms) across all years: 751

Tickers per year:
year
2012    497
2013    497
2014    499
2015    502
2016    506
2017    505
2018    505
2019    505
2020    505
2021    505
2022    503
2023    503
2024    503
Name: ticker, dtype: int64


In [8]:
# Check 1: GE removed from S&P 500 in June 2018
ge_years = panel[panel["ticker"] == "GE"]["year"].tolist()
print(f"GE present in years: {ge_years}")
# Expected: 2012-2017 (last full year before removal), 2018 questionable, NOT 2019+

# Check 2: Facebook → Meta ticker change October 2022
fb_years = panel[panel["ticker"] == "FB"]["year"].tolist()
meta_years = panel[panel["ticker"] == "META"]["year"].tolist()
print(f"FB present in years:   {fb_years}")
print(f"META present in years: {meta_years}")
# Expected: FB in 2012-2021, META in 2022+

# Check 3: Tesla added December 2020
tsla_years = panel[panel["ticker"] == "TSLA"]["year"].tolist()
print(f"TSLA present in years: {tsla_years}")
# Expected: 2020+ (no TSLA in 2019 or earlier)

# Check 4: Twitter removed November 2022 (Musk acquisition)
twtr_years = panel[panel["ticker"] == "TWTR"]["year"].tolist()
print(f"TWTR present in years: {twtr_years}")
# Expected: 2013-2021, NOT 2022+

GE present in years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
FB present in years:   [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
META present in years: [2022, 2023, 2024]
TSLA present in years: [2020, 2021, 2022, 2023, 2024]
TWTR present in years: [2018, 2019, 2020, 2021]


In [9]:
# Look at every snapshot date where GE appears
ge_snapshots = exploded[exploded["ticker"] == "GE"]["date"].sort_values()
print(f"Number of snapshots with GE: {len(ge_snapshots)}")
print(f"First GE snapshot: {ge_snapshots.min().date()}")
print(f"Last GE snapshot:  {ge_snapshots.max().date()}")
print()
print("All snapshot dates with GE (showing first 10 and last 10):")
print(ge_snapshots.head(10).dt.date.tolist())
print("...")
print(ge_snapshots.tail(10).dt.date.tolist())
print()

# Also check for GE-related tickers in case it's a ticker change issue
ge_like = exploded[exploded["ticker"].str.startswith("GE", na=False)]["ticker"].unique()
print(f"All tickers starting with 'GE': {ge_like}")

Number of snapshots with GE: 879
First GE snapshot: 2012-01-03
Last GE snapshot:  2024-06-24

All snapshot dates with GE (showing first 10 and last 10):
[datetime.date(2012, 1, 3), datetime.date(2012, 1, 4), datetime.date(2012, 1, 12), datetime.date(2012, 1, 19), datetime.date(2012, 1, 20), datetime.date(2012, 1, 23), datetime.date(2012, 2, 2), datetime.date(2012, 2, 8), datetime.date(2012, 2, 9), datetime.date(2012, 2, 10)]
...
[datetime.date(2023, 9, 18), datetime.date(2023, 10, 2), datetime.date(2023, 10, 18), datetime.date(2024, 2, 1), datetime.date(2024, 3, 4), datetime.date(2024, 3, 18), datetime.date(2024, 3, 25), datetime.date(2024, 4, 3), datetime.date(2024, 5, 8), datetime.date(2024, 6, 24)]

All tickers starting with 'GE': ['GE' 'GEN' 'GEHC' 'GEV']


In [10]:
output_path = DATA_PROCESSED / "sp500_universe.parquet"
panel.to_parquet(output_path, index=False)

# Verify by reading it back
check = pd.read_parquet(output_path)
print(f"Saved {len(check):,} rows to {output_path}")
print(f"Columns: {list(check.columns)}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

Saved 6,535 rows to /Users/<wrds-username>/thesis/data/sp500_universe.parquet
Columns: ['year', 'ticker', 'snapshot_date']
File size: 12.2 KB


In [11]:
# Lehman Brothers (LEHMQ) — should NOT appear; collapsed in 2008, before sample
print(f"LEH or LEHMQ in panel: {panel['ticker'].isin(['LEH', 'LEHMQ']).any()}")

# Netflix (NFLX) — should be present throughout sample (added 2010)
nflx_years = panel[panel["ticker"] == "NFLX"]["year"].tolist()
print(f"NFLX years: {nflx_years}")

# Amazon (AMZN) — should be present throughout (added 2005)
amzn_years = panel[panel["ticker"] == "AMZN"]["year"].tolist()
print(f"AMZN years: {amzn_years}")

# Yahoo (AABA was Yahoo's ticker after rename in 2017) — should appear pre-Verizon-acquisition
aaba_years = panel[panel["ticker"] == "AABA"]["year"].tolist()
yhoo_years = panel[panel["ticker"] == "YHOO"]["year"].tolist()
print(f"YHOO years: {yhoo_years}")
print(f"AABA years: {aaba_years}")

LEH or LEHMQ in panel: False
NFLX years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
AMZN years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
YHOO years: []
AABA years: [2012, 2013, 2014, 2015, 2016]


In [12]:
# Confirm AAPL is present every year and only once per year
aapl = panel[panel["ticker"] == "AAPL"]
print(f"AAPL rows: {len(aapl)}, years: {aapl['year'].tolist()}")

# Check for duplicate (year, ticker) pairs — there should be NONE
duplicates = panel.duplicated(subset=["year", "ticker"]).sum()
print(f"Duplicate (year, ticker) pairs: {duplicates}")

# Check the average panel year count is plausible
avg = panel.groupby("year")["ticker"].nunique().mean()
print(f"Average tickers per year: {avg:.1f}")  # should be 502-ish

# Total firms across the sample window
total_firms = panel["ticker"].nunique()
print(f"Total distinct firms across 2012-2024: {total_firms}")

AAPL rows: 13, years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Duplicate (year, ticker) pairs: 0
Average tickers per year: 502.7
Total distinct firms across 2012-2024: 751
